In [6]:
import time
import numpy as np
import sklearn
from sklearn.neighbors import KDTree, BallTree, radius_neighbors_graph
from sklearn.metrics.pairwise import euclidean_distances
from scipy.sparse import csr_array
from scipy.io import mmwrite, mmread

import cppimport
import cppimport.import_hook
#from extensions.brute_force import _BruteForce

def csr_nng(neighs):
    n = len(neighs)
    rowptrs, colids, dists = [], [], []
    for i in range(n):
        rowptrs.append(len(colids))
        colids += list(neighs[0][i])
        dists += list(neighs[1][i])
    rowptrs.append(len(colids))
    return csr_array((dists, colids, rowptrs), shape=(n,n))
    

class RadiusNeighborsGraph(object):
    
    def __init__(self, points, method="balltree", metric="euclidean"):
        assert metric in ("euclidean", "cosine", "manhattan", 'l1', "l2")
        assert method in ("balltree", "kdtree", "covertree", "bruteforce")
        self.points = points
        self.method = method
        self.metric = metric
        
    def build_index(self, **kwargs):
        if self.method == "balltree":
            self.index = BallTree(self.points, metric=self.metric, **kwargs)
        elif self.method == "kdtree":
            self.index = KDTree(self.points, metric=self.metric, **kwargs)
        elif self.method == "bruteforce":
            self.index = BruteForce(self.points, metric=self.metric, **kwargs)
            
    def radius_neighbors_graph(self, radius, num_threads=1):
        if self.method == "balltree":
            return csr_nng(self.index.query_radius(self.points, return_distance=True, r=radius))
        elif self.method == "kdtree":
            return csr_nng(self.index.query_radius(self.points, return_distance=True, r=radius))
        elif self.method == "bruteforce":
            return self.index.query_radius(self.points, radius, num_threads)
            
        

        
        
n, d = 10000, 16
points = np.random.uniform(0,1,size=(n,d))

rng = RadiusNeighborsGraph(points, "balltree", "euclidean")
rng.build_index()

In [16]:
neighs = rng.radius_neighbors_graph(0.8)

In [21]:
neighs[0][0]

array([   0, 9261, 3426, 4238])

In [22]:
neighs[1][0]

array([0.    , 0.7801, 0.7359, 0.7866])